# Definition of done — scorecard

Scores the project against `docs/ROADMAP.md` section 7, the table the FYP is
graded on. Queries the live Hugging Face Space, so **no models load here** --
this is network-bound, not compute-bound, which is exactly why it belongs on
Colab rather than a home connection.

Takes roughly **40-60 minutes** for 248 gold + 100 out-of-scope queries.

Run all cells. The last one prints the table and writes
`eval/definition_of_done.md`. Progress checkpoints every 25 queries, so if the
runtime drops you can re-run and it resumes rather than starting over.

In [ ]:
!pip install -q gradio_client pandas 2>/dev/null
print("deps installed")

In [ ]:
!rm -rf naari-ai
!git clone -q --depth 1 -b sana/phase3 https://github.com/sana200420/naari-ai.git
%cd naari-ai
print("cloned sana/phase3")

In [ ]:
# Wake the Space first. A cold Space reloads ~7GB of weights, and the first
# query would otherwise time out and be recorded as a failure rather than a
# slow success.
import json, time
from gradio_client import Client

t = time.time()
client = Client("Sanapalijo/naari-ai")
d = json.loads(client.predict(query="حيض جي چڪر ڇا آهي؟", top_k=5, api_name="/retrieve"))
print(f"Space awake in {time.time()-t:.0f}s -- top1 id {d['results'][0]['answer_id']}")

In [ ]:
!python scripts/definition_of_done.py

In [ ]:
# Re-run this cell if the runtime dropped mid-way. It resumes from the
# checkpoint instead of repeating the queries already done.
!python scripts/definition_of_done.py

In [ ]:
from google.colab import files
print(open("eval/definition_of_done.md", encoding="utf-8").read())
files.download("eval/definition_of_done.md")

## After it finishes

Download `eval/definition_of_done.md` and send it over, or commit it:

```bash
git add eval/definition_of_done.md && git commit -m "eval: definition-of-done scorecard" && git push
```

Two targets are deliberately absent from the table, and the script says so
rather than quietly omitting them:

- **Danger-sign recall** and **false-escalation** belong to `api/safety` and
  have their own harness (`eval/run_danger_gate_eval.py`). Two implementations
  of one number can only disagree.
- **Faithfulness** needs a human reading answers against their sources. A
  script claiming to measure it would be worse than recording it as
  unmeasured.